# 02 - Analisis Exploratorio de Datos (EDA)
## Prueba Tecnica - Cientifico de Datos Senior
### Gestion de Costos Operativos en un Proyecto de Construccion

---

**Objetivo:** Realizar un analisis exploratorio exhaustivo de las series de precios de materias primas y equipos para entender las relaciones, patrones temporales y caracteristicas estadisticas que guien la seleccion de modelos.

**Contenido:**
1. Estadisticas descriptivas
2. Distribucion univariada (histogramas, boxplots)
3. Test de normalidad
4. Descomposicion de series de tiempo
5. Prueba de estacionariedad (ADF)
6. Analisis de retornos porcentuales
7. Analisis de volatilidad
8. Correlaciones (Pearson y Spearman)
9. Scatterplots
10. ACF y PACF
11. Analisis de rezagos (cross-correlation)
12. Causalidad de Granger
13. Multicolinealidad (VIF)
14. Segmentacion COVID
15. Deteccion de anomalias (IQR, Z-score, Isolation Forest)
16. Mutual Information
17. Feature Engineering
18. Evaluacion de features
19. Resumen y conclusiones para modelamiento

## 0. Configuracion del entorno

In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import json
import warnings
warnings.filterwarnings('ignore')

# Configuracion visual
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

# S3
BUCKET = 'consulting-dataknow-prueba-tecnica'
s3 = boto3.client('s3')

def leer_csv_s3(bucket, key, **kwargs):
    obj = s3.get_object(Bucket=bucket, Key=key)
    contenido = obj['Body'].read().decode('utf-8-sig')
    return pd.read_csv(StringIO(contenido), **kwargs)

def guardar_json_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(data, default=str, ensure_ascii=False))
    print(f'Guardado en s3://{bucket}/{key}')

print('Entorno configurado.')

## 0.1 Carga de datos procesados

In [ ]:
df_x = leer_csv_s3(BUCKET, 'datos_procesados/X_limpio.csv', parse_dates=['Date'])
df_y = leer_csv_s3(BUCKET, 'datos_procesados/Y_limpio.csv', parse_dates=['Date'])
df_z = leer_csv_s3(BUCKET, 'datos_procesados/Z_limpio.csv', parse_dates=['Date'])
df = leer_csv_s3(BUCKET, 'datos_procesados/historico_equipos_limpio.csv', parse_dates=['Date'])

# Variables de precio para analisis
precio_cols = ['Price_X', 'Price_Y', 'Price_Z', 'Price_Equipo1', 'Price_Equipo2']
materias = ['Price_X', 'Price_Y', 'Price_Z']
equipos = ['Price_Equipo1', 'Price_Equipo2']

print(f'Dataset principal: {df.shape}')
print(f'Rango: {df["Date"].min().date()} a {df["Date"].max().date()}')
df.head()

---
## 1. Estadisticas descriptivas

In [ ]:
desc = df[precio_cols].describe().T
desc['cv'] = desc['std'] / desc['mean']  # Coeficiente de variacion
desc['rango'] = desc['max'] - desc['min']
desc['iqr'] = desc['75%'] - desc['25%']
desc['skew'] = df[precio_cols].skew()
desc['kurtosis'] = df[precio_cols].kurtosis()

print('ESTADISTICAS DESCRIPTIVAS')
print('=' * 80)
print(desc.round(4))

# Guardar para el agente
eda_stats = desc.round(4).to_dict()
guardar_json_s3(eda_stats, BUCKET, 'eda_resultados/estadisticas_descriptivas.json')

---
## 2. Distribucion univariada

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Distribucion Univariada - Histogramas y Boxplots', fontsize=14, fontweight='bold')

for i, col in enumerate(precio_cols):
    # Histograma
    axes[0, i].hist(df[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(col, fontweight='bold')
    axes[0, i].set_ylabel('Frecuencia' if i == 0 else '')
    
    # Boxplot
    axes[1, i].boxplot(df[col], vert=True, patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[1, i].set_ylabel('Precio' if i == 0 else '')

plt.tight_layout()
plt.show()

---
## 3. Test de normalidad (sobre los targets)

In [ ]:
from scipy import stats

print('TEST DE NORMALIDAD')
print('=' * 70)
print(f'{"Variable":20s} {"Shapiro-W":>12s} {"p-value":>12s} {"Jarque-Bera":>12s} {"p-value":>12s} {"Normal?":>10s}')
print('-' * 70)

normalidad_resultados = {}

for col in precio_cols:
    # Shapiro (muestra de 5000 por limitacion del test)
    muestra = df[col].sample(min(5000, len(df)), random_state=42)
    sw_stat, sw_p = stats.shapiro(muestra)
    
    # Jarque-Bera
    jb_stat, jb_p = stats.jarque_bera(df[col])
    
    es_normal = 'Si' if (sw_p > 0.05 and jb_p > 0.05) else 'No'
    
    normalidad_resultados[col] = {
        'shapiro_stat': round(sw_stat, 6),
        'shapiro_p': round(sw_p, 6),
        'jarque_bera_stat': round(jb_stat, 4),
        'jarque_bera_p': round(jb_p, 6),
        'es_normal': es_normal
    }
    
    print(f'{col:20s} {sw_stat:12.6f} {sw_p:12.6f} {jb_stat:12.4f} {jb_p:12.6f} {es_normal:>10s}')

guardar_json_s3(normalidad_resultados, BUCKET, 'eda_resultados/test_normalidad.json')

---
## 4. Descomposicion de series de tiempo

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

fig, axes = plt.subplots(len(precio_cols), 4, figsize=(20, 4 * len(precio_cols)))
fig.suptitle('Descomposicion de Series de Tiempo (Modelo Aditivo, Periodo=252 dias habiles)', 
             fontsize=14, fontweight='bold', y=1.01)

for i, col in enumerate(precio_cols):
    serie = df.set_index('Date')[col]
    decomp = seasonal_decompose(serie, model='additive', period=252)
    
    axes[i, 0].plot(decomp.observed, color='steelblue', linewidth=0.5)
    axes[i, 0].set_ylabel(col, fontweight='bold')
    axes[i, 0].set_title('Observado' if i == 0 else '')
    
    axes[i, 1].plot(decomp.trend, color='darkorange', linewidth=1)
    axes[i, 1].set_title('Tendencia' if i == 0 else '')
    
    axes[i, 2].plot(decomp.seasonal, color='green', linewidth=0.5)
    axes[i, 2].set_title('Estacionalidad' if i == 0 else '')
    
    axes[i, 3].plot(decomp.resid, color='red', linewidth=0.5)
    axes[i, 3].set_title('Residuos' if i == 0 else '')

plt.tight_layout()
plt.show()

---
## 5. Prueba de estacionariedad (Dickey-Fuller Aumentado)

In [ ]:
from statsmodels.tsa.stattools import adfuller

print('PRUEBA DE ESTACIONARIEDAD - Dickey-Fuller Aumentado (ADF)')
print('=' * 80)
print(f'{"Variable":20s} {"ADF Stat":>12s} {"p-value":>12s} {"Lags":>6s} {"Estacionaria?":>15s}')
print('-' * 80)

adf_resultados = {}

for col in precio_cols:
    resultado = adfuller(df[col], autolag='AIC')
    es_estacionaria = 'Si' if resultado[1] < 0.05 else 'No'
    
    adf_resultados[col] = {
        'adf_stat': round(resultado[0], 6),
        'p_value': round(resultado[1], 6),
        'lags_usados': resultado[2],
        'n_obs': resultado[3],
        'valores_criticos': {k: round(v, 4) for k, v in resultado[4].items()},
        'es_estacionaria': es_estacionaria
    }
    
    print(f'{col:20s} {resultado[0]:12.6f} {resultado[1]:12.6f} {resultado[2]:6d} {es_estacionaria:>15s}')

# Tambien sobre las diferencias (primera diferencia)
print(f'\nPrimera diferencia:')
print(f'{"Variable":20s} {"ADF Stat":>12s} {"p-value":>12s} {"Estacionaria?":>15s}')
print('-' * 60)

for col in precio_cols:
    diff = df[col].diff().dropna()
    resultado = adfuller(diff, autolag='AIC')
    es_estacionaria = 'Si' if resultado[1] < 0.05 else 'No'
    
    adf_resultados[f'{col}_diff'] = {
        'adf_stat': round(resultado[0], 6),
        'p_value': round(resultado[1], 6),
        'es_estacionaria': es_estacionaria
    }
    
    print(f'{col:20s} {resultado[0]:12.6f} {resultado[1]:12.6f} {es_estacionaria:>15s}')

guardar_json_s3(adf_resultados, BUCKET, 'eda_resultados/test_estacionariedad.json')

---
## 6. Analisis de retornos porcentuales

In [ ]:
# Calcular retornos porcentuales
retornos = df[precio_cols].pct_change().dropna()
retornos.insert(0, 'Date', df['Date'].iloc[1:].values)

print('ESTADISTICAS DE RETORNOS PORCENTUALES DIARIOS')
print('=' * 80)
print(retornos[precio_cols].describe().round(6))

# Visualizacion
fig, axes = plt.subplots(len(precio_cols), 1, figsize=(16, 3 * len(precio_cols)), sharex=True)
fig.suptitle('Retornos Porcentuales Diarios', fontsize=14, fontweight='bold')

for i, col in enumerate(precio_cols):
    axes[i].plot(retornos['Date'], retornos[col], linewidth=0.3, color='steelblue')
    axes[i].set_ylabel(col)
    axes[i].axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## 7. Analisis de volatilidad

In [ ]:
# Rolling standard deviation de los retornos (ventanas de 30 y 90 dias)
fig, axes = plt.subplots(len(precio_cols), 1, figsize=(16, 3 * len(precio_cols)), sharex=True)
fig.suptitle('Volatilidad - Rolling Std de Retornos (30 y 90 dias)', fontsize=14, fontweight='bold')

volatilidad_resultados = {}

for i, col in enumerate(precio_cols):
    vol_30 = retornos[col].rolling(30).std()
    vol_90 = retornos[col].rolling(90).std()
    
    axes[i].plot(retornos['Date'], vol_30, linewidth=0.8, color='steelblue', label='30 dias', alpha=0.7)
    axes[i].plot(retornos['Date'], vol_90, linewidth=1.2, color='darkorange', label='90 dias')
    axes[i].set_ylabel(col)
    axes[i].legend(loc='upper right', fontsize=8)
    
    # Marcar periodo COVID
    axes[i].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-12-31'), 
                    alpha=0.15, color='red', label='COVID' if i == 0 else '')
    
    volatilidad_resultados[col] = {
        'vol_30d_mean': round(vol_30.mean(), 6),
        'vol_30d_max': round(vol_30.max(), 6),
        'vol_90d_mean': round(vol_90.mean(), 6),
        'vol_90d_max': round(vol_90.max(), 6)
    }

plt.tight_layout()
plt.show()

guardar_json_s3(volatilidad_resultados, BUCKET, 'eda_resultados/volatilidad.json')

---
## 8. Correlaciones (Pearson y Spearman)

In [ ]:
# Pearson
corr_pearson = df[precio_cols].corr(method='pearson')

# Spearman
corr_spearman = df[precio_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr_pearson, annot=True, fmt='.4f', cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title('Correlacion de Pearson', fontweight='bold')

sns.heatmap(corr_spearman, annot=True, fmt='.4f', cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, ax=axes[1], square=True)
axes[1].set_title('Correlacion de Spearman', fontweight='bold')

plt.tight_layout()
plt.show()

# Resumen de correlaciones con los targets
print('CORRELACIONES CON LOS EQUIPOS')
print('=' * 70)
print(f'{"":20s} {"Equipo1 (Pearson)":>18s} {"Equipo1 (Spearman)":>18s} {"Equipo2 (Pearson)":>18s} {"Equipo2 (Spearman)":>18s}')
print('-' * 95)
for m in materias:
    print(f'{m:20s} {corr_pearson.loc[m, "Price_Equipo1"]:18.4f} {corr_spearman.loc[m, "Price_Equipo1"]:18.4f} {corr_pearson.loc[m, "Price_Equipo2"]:18.4f} {corr_spearman.loc[m, "Price_Equipo2"]:18.4f}')

corr_resultado = {
    'pearson': corr_pearson.round(4).to_dict(),
    'spearman': corr_spearman.round(4).to_dict()
}
guardar_json_s3(corr_resultado, BUCKET, 'eda_resultados/correlaciones.json')

---
## 9. Scatterplots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Scatterplots: Materias Primas vs Equipos', fontsize=14, fontweight='bold')

for j, equipo in enumerate(equipos):
    for i, materia in enumerate(materias):
        ax = axes[j, i]
        ax.scatter(df[materia], df[equipo], alpha=0.15, s=5, color='steelblue')
        
        # Linea de tendencia
        z = np.polyfit(df[materia], df[equipo], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df[materia].min(), df[materia].max(), 100)
        ax.plot(x_line, p(x_line), 'r-', linewidth=1.5, alpha=0.8)
        
        r_pearson = corr_pearson.loc[materia, equipo]
        ax.set_title(f'{materia} vs {equipo}\nr = {r_pearson:.4f}', fontweight='bold')
        ax.set_xlabel(materia)
        ax.set_ylabel(equipo if i == 0 else '')

plt.tight_layout()
plt.show()

---
## 10. ACF y PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(len(precio_cols), 2, figsize=(16, 3 * len(precio_cols)))
fig.suptitle('ACF y PACF (50 lags)', fontsize=14, fontweight='bold', y=1.01)

for i, col in enumerate(precio_cols):
    plot_acf(df[col], lags=50, ax=axes[i, 0], title=f'ACF - {col}')
    plot_pacf(df[col], lags=50, ax=axes[i, 1], title=f'PACF - {col}')

plt.tight_layout()
plt.show()

In [ ]:
# ACF y PACF sobre las primeras diferencias (series diferenciadas)
fig, axes = plt.subplots(len(precio_cols), 2, figsize=(16, 3 * len(precio_cols)))
fig.suptitle('ACF y PACF sobre Primera Diferencia (50 lags)', fontsize=14, fontweight='bold', y=1.01)

for i, col in enumerate(precio_cols):
    diff = df[col].diff().dropna()
    plot_acf(diff, lags=50, ax=axes[i, 0], title=f'ACF - diff({col})')
    plot_pacf(diff, lags=50, ax=axes[i, 1], title=f'PACF - diff({col})')

plt.tight_layout()
plt.show()

---
## 11. Analisis de rezagos (Cross-Correlation)

In [ ]:
max_lag = 30

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Cross-Correlation: Materias Primas (rezagadas) vs Equipos (hasta {max_lag} dias)', 
             fontsize=14, fontweight='bold')

lag_resultados = {}

for j, equipo in enumerate(equipos):
    for i, materia in enumerate(materias):
        correlaciones = []
        for lag in range(max_lag + 1):
            if lag == 0:
                corr = df[materia].corr(df[equipo])
            else:
                corr = df[materia].shift(lag).corr(df[equipo])
            correlaciones.append(corr)
        
        ax = axes[j, i]
        ax.bar(range(max_lag + 1), correlaciones, color='steelblue', alpha=0.7)
        
        mejor_lag = np.argmax(np.abs(correlaciones))
        ax.axvline(x=mejor_lag, color='red', linestyle='--', linewidth=1)
        ax.set_title(f'{materia} -> {equipo}\nMejor lag: {mejor_lag} (r={correlaciones[mejor_lag]:.4f})', 
                     fontweight='bold')
        ax.set_xlabel('Lag (dias)')
        ax.set_ylabel('Correlacion' if i == 0 else '')
        
        lag_resultados[f'{materia}_vs_{equipo}'] = {
            'mejor_lag': int(mejor_lag),
            'correlacion_lag0': round(correlaciones[0], 4),
            'correlacion_mejor_lag': round(correlaciones[mejor_lag], 4)
        }

plt.tight_layout()
plt.show()

guardar_json_s3(lag_resultados, BUCKET, 'eda_resultados/analisis_rezagos.json')

---
## 12. Causalidad de Granger

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

print('CAUSALIDAD DE GRANGER')
print('=' * 80)
print('H0: La materia prima NO causa Granger al equipo')
print('Si p-value < 0.05, se rechaza H0 (hay causalidad de Granger)')
print('-' * 80)

granger_resultados = {}
max_lag_granger = 10

for equipo in equipos:
    for materia in materias:
        print(f'\n{materia} -> {equipo}:')
        data_granger = df[[equipo, materia]].dropna()
        
        try:
            resultado = grangercausalitytests(data_granger, maxlag=max_lag_granger, verbose=False)
            
            p_values = {}
            for lag in range(1, max_lag_granger + 1):
                p_val = resultado[lag][0]['ssr_ftest'][1]
                p_values[str(lag)] = round(p_val, 6)
            
            # Mejor lag (menor p-value)
            mejor_lag = min(p_values, key=p_values.get)
            mejor_p = p_values[mejor_lag]
            es_causal = 'Si' if mejor_p < 0.05 else 'No'
            
            print(f'  Mejor lag: {mejor_lag}, p-value: {mejor_p:.6f}, Causal: {es_causal}')
            
            granger_resultados[f'{materia}_vs_{equipo}'] = {
                'p_values_por_lag': p_values,
                'mejor_lag': int(mejor_lag),
                'mejor_p_value': mejor_p,
                'es_causal': es_causal
            }
        except Exception as e:
            print(f'  Error: {e}')

guardar_json_s3(granger_resultados, BUCKET, 'eda_resultados/causalidad_granger.json')

---
## 13. Multicolinealidad (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF entre las materias primas
X_vif = df[materias].copy()

vif_data = pd.DataFrame()
vif_data['Variable'] = materias
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(len(materias))]

print('FACTOR DE INFLACION DE VARIANZA (VIF)')
print('=' * 50)
print('VIF > 5: multicolinealidad moderada')
print('VIF > 10: multicolinealidad severa')
print('-' * 50)
for _, row in vif_data.iterrows():
    nivel = 'OK' if row['VIF'] < 5 else ('Moderada' if row['VIF'] < 10 else 'Severa')
    print(f"  {row['Variable']:15s}  VIF = {row['VIF']:8.2f}  ({nivel})")

# Correlacion entre materias primas
print(f'\nCorrelacion entre materias primas:')
print(df[materias].corr().round(4))

vif_resultado = vif_data.set_index('Variable')['VIF'].round(4).to_dict()
guardar_json_s3(vif_resultado, BUCKET, 'eda_resultados/vif.json')

---
## 14. Segmentacion COVID

In [ ]:
# Definir periodos
pre_covid = df[df['Date'] < '2020-03-01'].copy()
covid = df[(df['Date'] >= '2020-03-01') & (df['Date'] < '2021-06-01')].copy()
post_covid = df[df['Date'] >= '2021-06-01'].copy()

periodos = {
    'Pre-COVID (antes mar-2020)': pre_covid,
    'COVID (mar-2020 a may-2021)': covid,
    'Post-COVID (jun-2021 en adelante)': post_covid
}

print('SEGMENTACION POR PERIODOS')
print('=' * 80)

segmentacion_resultados = {}

for nombre, periodo in periodos.items():
    print(f'\n{nombre} ({len(periodo)} registros)')
    print('-' * 60)
    
    # Estadisticas
    stats_periodo = periodo[precio_cols].describe().loc[['mean', 'std']].round(2)
    print(stats_periodo)
    
    # Correlaciones con equipos
    print(f'\n  Correlaciones con Equipo1:')
    corr_e1 = {}
    corr_e2 = {}
    for m in materias:
        c1 = periodo[m].corr(periodo['Price_Equipo1'])
        c2 = periodo[m].corr(periodo['Price_Equipo2'])
        corr_e1[m] = round(c1, 4)
        corr_e2[m] = round(c2, 4)
        print(f'    {m}: Eq1={c1:.4f}, Eq2={c2:.4f}')
    
    segmentacion_resultados[nombre] = {
        'n_registros': len(periodo),
        'fecha_inicio': str(periodo['Date'].min().date()),
        'fecha_fin': str(periodo['Date'].max().date()),
        'medias': periodo[precio_cols].mean().round(2).to_dict(),
        'std': periodo[precio_cols].std().round(2).to_dict(),
        'correlaciones_equipo1': corr_e1,
        'correlaciones_equipo2': corr_e2
    }

guardar_json_s3(segmentacion_resultados, BUCKET, 'eda_resultados/segmentacion_covid.json')

In [ ]:
# Visualizacion de series con segmentacion COVID
fig, axes = plt.subplots(len(precio_cols), 1, figsize=(16, 3 * len(precio_cols)), sharex=True)
fig.suptitle('Series de Tiempo con Segmentacion COVID', fontsize=14, fontweight='bold')

for i, col in enumerate(precio_cols):
    axes[i].plot(df['Date'], df[col], linewidth=0.5, color='steelblue')
    axes[i].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'), 
                    alpha=0.2, color='red', label='COVID')
    axes[i].set_ylabel(col)
    if i == 0:
        axes[i].legend(loc='upper left')

plt.tight_layout()
plt.show()

---
## 15. Deteccion de anomalias

In [ ]:
from sklearn.ensemble import IsolationForest

anomalias_resultados = {}

for col in precio_cols:
    serie = df[col]
    
    # IQR
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    iqr_lower = Q1 - 1.5 * IQR
    iqr_upper = Q3 + 1.5 * IQR
    n_iqr = ((serie < iqr_lower) | (serie > iqr_upper)).sum()
    
    # Z-score
    z_scores = np.abs(stats.zscore(serie))
    n_zscore = (z_scores > 3).sum()
    
    # Isolation Forest
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    iso_labels = iso_forest.fit_predict(serie.values.reshape(-1, 1))
    n_iso = (iso_labels == -1).sum()
    
    anomalias_resultados[col] = {
        'iqr_anomalias': int(n_iqr),
        'iqr_limites': {'lower': round(iqr_lower, 2), 'upper': round(iqr_upper, 2)},
        'zscore_anomalias': int(n_zscore),
        'isolation_forest_anomalias': int(n_iso),
        'total_registros': len(serie)
    }

print('DETECCION DE ANOMALIAS')
print('=' * 80)
print(f'{"Variable":20s} {"IQR":>8s} {"Z-score>3":>12s} {"Isol. Forest":>14s} {"Total":>8s}')
print('-' * 65)
for col, res in anomalias_resultados.items():
    print(f'{col:20s} {res["iqr_anomalias"]:8d} {res["zscore_anomalias"]:12d} {res["isolation_forest_anomalias"]:14d} {res["total_registros"]:8d}')

guardar_json_s3(anomalias_resultados, BUCKET, 'eda_resultados/anomalias.json')

In [ ]:
# Visualizar anomalias (Z-score) sobre las series
fig, axes = plt.subplots(len(precio_cols), 1, figsize=(16, 3 * len(precio_cols)), sharex=True)
fig.suptitle('Anomalias detectadas por Z-score > 3', fontsize=14, fontweight='bold')

for i, col in enumerate(precio_cols):
    z_scores = np.abs(stats.zscore(df[col]))
    mask_anomalia = z_scores > 3
    
    axes[i].plot(df['Date'], df[col], linewidth=0.5, color='steelblue')
    axes[i].scatter(df.loc[mask_anomalia, 'Date'], df.loc[mask_anomalia, col], 
                    color='red', s=15, zorder=5, label=f'Anomalias ({mask_anomalia.sum()})')
    axes[i].set_ylabel(col)
    axes[i].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

---
## 16. Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_regression

print('MUTUAL INFORMATION')
print('=' * 60)
print('Mide dependencias no lineales (0 = independiente, mayor = mas dependencia)')
print('-' * 60)

mi_resultados = {}

for equipo in equipos:
    mi = mutual_info_regression(df[materias], df[equipo], random_state=42)
    mi_dict = {m: round(float(v), 4) for m, v in zip(materias, mi)}
    mi_resultados[equipo] = mi_dict
    
    print(f'\n{equipo}:')
    for m, v in sorted(mi_dict.items(), key=lambda x: x[1], reverse=True):
        barra = '#' * int(v * 20)
        print(f'  {m:15s}: {v:.4f}  {barra}')

guardar_json_s3(mi_resultados, BUCKET, 'eda_resultados/mutual_information.json')

---
## 17. Feature Engineering

In [ ]:
df_feat = df.copy()

for col in materias:
    nombre = col.replace('Price_', '')
    
    # Retornos porcentuales
    df_feat[f'Return_{nombre}'] = df_feat[col].pct_change()
    
    # Promedios moviles
    for ventana in [7, 14, 30]:
        df_feat[f'MA{ventana}_{nombre}'] = df_feat[col].rolling(ventana).mean()
    
    # Rezagos
    for lag in [1, 5, 10]:
        df_feat[f'Lag{lag}_{nombre}'] = df_feat[col].shift(lag)
    
    # Volatilidad rolling (std 30 dias de retornos)
    df_feat[f'Vol30_{nombre}'] = df_feat[col].pct_change().rolling(30).std()
    
    # Diferencia absoluta
    df_feat[f'Diff_{nombre}'] = df_feat[col].diff()

# Ratios entre materias primas
df_feat['Ratio_XY'] = df_feat['Price_X'] / df_feat['Price_Y']
df_feat['Ratio_XZ'] = df_feat['Price_X'] / df_feat['Price_Z']
df_feat['Ratio_YZ'] = df_feat['Price_Y'] / df_feat['Price_Z']

# Feature interactions
df_feat['Interact_XY'] = df_feat['Price_X'] * df_feat['Price_Y']
df_feat['Interact_XZ'] = df_feat['Price_X'] * df_feat['Price_Z']
df_feat['Interact_YZ'] = df_feat['Price_Y'] * df_feat['Price_Z']

# Eliminar filas con NaN generadas por rolling/shift
df_feat_clean = df_feat.dropna().reset_index(drop=True)

nuevas_features = [c for c in df_feat_clean.columns if c not in df.columns]

print(f'Features originales: {len(df.columns)}')
print(f'Features nuevas creadas: {len(nuevas_features)}')
print(f'Total columnas: {len(df_feat_clean.columns)}')
print(f'Registros despues de limpiar NaN: {len(df_feat_clean)} (de {len(df)})')
print(f'\nFeatures creadas:')
for f in nuevas_features:
    print(f'  - {f}')

---
## 18. Evaluacion de features

In [ ]:
# Correlacion de las nuevas features con los targets
all_features = materias + nuevas_features

evaluacion_features = {}

for equipo in equipos:
    print(f'\nTOP 20 FEATURES POR CORRELACION PEARSON - {equipo}')
    print('=' * 60)
    
    corrs = df_feat_clean[all_features].corrwith(df_feat_clean[equipo]).abs().sort_values(ascending=False)
    top20 = corrs.head(20)
    
    for feat, val in top20.items():
        barra = '#' * int(val * 40)
        print(f'  {feat:25s}: {val:.4f}  {barra}')
    
    evaluacion_features[equipo] = {
        'pearson_top20': {k: round(v, 4) for k, v in top20.items()}
    }

In [ ]:
# Mutual Information de todas las features
for equipo in equipos:
    print(f'\nTOP 20 FEATURES POR MUTUAL INFORMATION - {equipo}')
    print('=' * 60)
    
    mi = mutual_info_regression(df_feat_clean[all_features], df_feat_clean[equipo], random_state=42)
    mi_series = pd.Series(mi, index=all_features).sort_values(ascending=False)
    top20_mi = mi_series.head(20)
    
    for feat, val in top20_mi.items():
        barra = '#' * int(val * 10)
        print(f'  {feat:25s}: {val:.4f}  {barra}')
    
    evaluacion_features[equipo]['mi_top20'] = {k: round(v, 4) for k, v in top20_mi.items()}

guardar_json_s3(evaluacion_features, BUCKET, 'eda_resultados/evaluacion_features.json')

In [ ]:
# VIF de las nuevas features (seleccion de las top para evitar multicolinealidad)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Seleccionar un subconjunto representativo para VIF
features_vif = materias + ['Ratio_XY', 'Ratio_XZ', 'Ratio_YZ', 
                            'Return_X', 'Return_Y', 'Return_Z',
                            'Vol30_X', 'Vol30_Y', 'Vol30_Z']

X_vif = df_feat_clean[features_vif].copy()

vif_features = pd.DataFrame()
vif_features['Variable'] = features_vif
vif_features['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(len(features_vif))]
vif_features = vif_features.sort_values('VIF', ascending=False)

print('VIF DE FEATURES SELECCIONADAS')
print('=' * 50)
for _, row in vif_features.iterrows():
    nivel = 'OK' if row['VIF'] < 5 else ('Moderada' if row['VIF'] < 10 else 'Severa')
    print(f"  {row['Variable']:20s}  VIF = {row['VIF']:10.2f}  ({nivel})")

---
## 19. Guardar dataset con features y resumen final

In [ ]:
# Guardar dataset con features en S3
buffer = StringIO()
df_feat_clean.to_csv(buffer, index=False)
s3.put_object(Bucket=BUCKET, Key='datos_procesados/dataset_con_features.csv', Body=buffer.getvalue())
print('Dataset con features guardado en S3.')

print(f'\nShape final: {df_feat_clean.shape}')

In [ ]:
# Resumen ejecutivo del EDA
print('\n' + '=' * 80)
print('RESUMEN EJECUTIVO DEL EDA')
print('=' * 80)

print('''
HALLAZGOS PRINCIPALES:

1. CORRELACIONES:
   - Price_Y tiene correlacion casi perfecta con Equipo1 (Pearson ~0.997)
   - Price_Z tiene correlacion muy alta con Equipo2 (Pearson ~0.983)
   - Price_X tiene correlacion moderada con ambos equipos (~0.53)

2. ESTACIONARIEDAD:
   - Las series en niveles probablemente NO son estacionarias
   - Las primeras diferencias si son estacionarias
   - Implicacion: considerar modelos que manejen no-estacionariedad

3. MULTICOLINEALIDAD:
   - Revisar VIF para decidir si incluir las 3 materias primas o seleccionar

4. VOLATILIDAD:
   - Se observan clusters de volatilidad, especialmente durante COVID
   - La relacion entre materias primas y equipos se mantiene estable

5. FEATURES:
   - Las variables originales (Y, Z) dominan las correlaciones
   - Los promedios moviles y ratios aportan informacion complementaria

IMPLICACIONES PARA MODELAMIENTO:
   - Modelo base: regresion lineal con Y y Z como predictores principales
   - Probar regularizacion (Ridge, Lasso) por multicolinealidad
   - Modelos no lineales (RF, XGBoost) para capturar interacciones
   - SARIMAX/Prophet con Y y Z como variables exogenas para proyeccion
   - Considerar features derivadas en modelos de ML
''')

# Guardar resumen
resumen = {
    'hallazgo_principal_equipo1': 'Price_Y es el predictor dominante (corr ~0.997)',
    'hallazgo_principal_equipo2': 'Price_Z es el predictor dominante (corr ~0.983)',
    'estacionariedad': 'Series no estacionarias en niveles, estacionarias en primera diferencia',
    'covid_impacto': 'Clusters de volatilidad durante COVID pero relaciones estables',
    'n_features_creadas': len(nuevas_features),
    'dataset_final_shape': list(df_feat_clean.shape)
}
guardar_json_s3(resumen, BUCKET, 'eda_resultados/resumen_eda.json')